In [ ]:
import sys
# Use an absolute path or relative path to the directory
sys.path.append("scripts")

import numpy as np
import pdb_voxelizier
import cnn_mlp_encoder
import jw_quantum_mapper
from scipy.optimize import minimize
from scipy.spatial.distance import squareform
from scipy.linalg import eigh
import sympy
import openfermion as op
import torch
import qiskit
import qiskit_algorithms
import qiskit_aer as q_aer

In [ ]:
num_sites = 6
tensor = pdb_voxelizier.pdb_to_tensor('proteins/1ENH.pdb')
coefficients = cnn_mlp_encoder.get_hamiltonian(tensor, num_qubits=num_sites)
qubit_instructions = jw_quantum_mapper.apply_jw(coefficients,num_sites=num_sites)


In [ ]:
# View instructions by uncommenting this
jw_quantum_mapper.display_instructions(qubit_instructions)

In [ ]:
import pyvista as pv
def visualize_tensor(tensor):
    # [batch_size, channels, Depth, Height, Width]
    protein = tensor[0]
    
    protein = protein.to_dense().max(axis=0).values

    array = protein.numpy()
    grid = pv.wrap(array) # Automatically recognizes as a dataset
    grid.plot(jupyter_backend="client",volume=True)

torch_tensor = torch.from_numpy(tensor)
visualize_tensor(torch_tensor)

Test our hamiltonian through a VQE!

In [ ]:
# # To convert our list of strings into a PauliSum, we need to loop through each instruction
def convert_qubit_operators_to_pauli_operators(qubit_operators:list):
    coef_list = []
    op_list = []
    multi_qubit_operator_set = set()
    for i in qubit_operators:
        # Split up the operation into coefficient and the gates
        operation_list = i.split("*")
        
        # We grab the coefficient as a float and each operator as a single string   
        coef = float(operation_list[0])
        operators = operation_list[1].strip().split(" ")

        # We add all coefficients into a list
        coef_list.append(coef)

        
        # Go through each operator in the list, convert to Qiskit 'Pauli' term
        # First, we create a list of identity terms since each Pauli needs to be the same dimension
        start_op_list = list("I" * (num_sites))
        
        # We want to not add even-numbered Y amount of observables due to creating zero gradients
        # y_count = None
        for term in operators:
            # if term[0] == "Y":
            #     if y_count is None:
            #         y_count = 0
            #     y_count += 1
            # First term is letter, next is integer
            start_op_list[int(term[1])] = term[0]
        # Additionally, we can also define our multi-qubit operators here! (Note, some single-qubit operators are here too but we deal with that later!)
        fin_str = "".join(start_op_list)
        # print(y_count)
        # if y_count is None or y_count % 2 == 1:
        multi_qubit_operator_set.add(fin_str)
        op_list.append(fin_str)
    
    return qiskit.quantum_info.SparsePauliOp(data=op_list,coeffs=coef_list), multi_qubit_operator_set

pauli_op, multi_qubit_operator = convert_qubit_operators_to_pauli_operators(qubit_instructions)

In [ ]:
print(multi_qubit_operator)

Get the ground state energy classically

In [ ]:
# To define the matrix, we convert the pauli operator directly into a matrix
hamiltonian_matrix = pauli_op.to_matrix()

print("\nFinal Hamiltonian Matrix:")
print(hamiltonian_matrix)

# Now, we use scipy.linalg.eigh to calculate the ground state value of this matrix
lowest = min(eigh(hamiltonian_matrix)[0])

print(f"\nGround State Energy: {lowest}")


Estimate ground state energy using VQE

In [ ]:
# Now, lets create our ansatz circuit. In this case I use a variation of the hardware efficient Ansatz (HEA), specifically, qiskit's efficient_su2
from qiskit_algorithms.minimum_eigensolvers import AdaptVQE
from qiskit.circuit.library import EvolvedOperatorAnsatz

n = pauli_op.num_qubits
layers = 3
ansatz = qiskit.circuit.library.efficient_su2(n, su2_gates=["ry"], entanglement="circular",reps=layers)
# ansatz = EvolvedOperatorAnsatz(name="Ansatz_Adapt",reps=10)
# num_params = ansatz.num_parameters
# print(f"This ansatz has {num_params} parameters.")
ansatz.decompose().draw("mpl",style="iqp")

Now we can run our circuit and attempt to converge on the ground state energy of the PauliSum.

In [ ]:
from qiskit_algorithms.optimizers import COBYLA
# Define Simulation
iterations = 1000
cobyla = COBYLA(maxiter=1000)
counts = []
values = []

def store_intermediate_result(eval_count,parameters,mean,std):
    counts.append(eval_count)
    values.append(mean)

In [ ]:
from qiskit_algorithms.utils import algorithm_globals
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

seed = 170
algorithm_globals.random_seed = seed

noiseless_estimator = AerEstimator(options={"default_precision": 1e-2})


In [ ]:
from qiskit_algorithms import VQE
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.optimizers import SLSQP

vqe = VQE(estimator=noiseless_estimator,ansatz=ansatz,optimizer=cobyla,callback=store_intermediate_result)

result = vqe.compute_minimum_eigenvalue(operator=pauli_op)

print(result.eigenvalue)
print(lowest)

ADAPT-VQE testing

In [ ]:
from qiskit.quantum_info import SparsePauliOp
vqe = VQE(estimator=noiseless_estimator, ansatz=None,optimizer=cobyla,callback=store_intermediate_result)

# Create our single-qubit building blocks for ADAPT-VQE to build off of
# operator = SparsePauliOp()

# Lets visualize all the operators we create!
test_list = list()
operator_list = list()
def all_single_qubit_operators(num_qubits:int):
    for i in range(num_qubits):
        for term in ["X","Y","Z"]:
            init_str = ["I"] * num_qubits # Start with a blank dimensionally-consistent operator
            # Create an operator variation
            init_str[i] = term

            # All to test list
            test_list.append("".join(init_str))

            # # Convert to SparsePauliOp object and append to list
            # operator_list.append(SparsePauliOp("".join(init_str)))

# We already defined combined qubit operators!
def all_entangled_qubit_operators(num_qubits:int):
    for i in range(num_qubits):
        for term in range(i+1,num_qubits):
            init_str = ["I"] * num_qubits
            for comb in ["XYZ","XZY","ZYX","ZXY","YXZ","YZX"]:
                    
                pass
            pass    
        pass
    pass

all_single_qubit_operators(n)
for op in test_list:
    multi_qubit_operator.add(op)
    
list_multi_qubit_operators = list(multi_qubit_operator)

operators_list = list()
for item in list_multi_qubit_operators:
    operators_list.append(SparsePauliOp(item))

adapt_vqe = AdaptVQE(solver=vqe,operators=operators_list)

results = adapt_vqe.compute_minimum_eigenvalue(pauli_op)
print(f"Final Result: {results.eigenvalue}")
print(f"Actual Result: {lowest}")
print(results.optimal_circuit.draw())
# print(adapt_vqe.get('optimal_circuit').draw("mpl"))
# eigenvalue, _ = adapt_vqe.compute_minimum_eigenvalue(pauli_op)